# Generar los embeddings oficiales y armar `datos_examen.zip`
**SI3003 · Evaluación 2 · Material del docente** (no compartir con estudiantes)

Este notebook:
1. carga la carpeta `datos_examen/` (corpus, preguntas, `embedding_info.json`),
2. genera `corpus_embeddings.npy` y `question_embeddings.npy` con el **modelo oficial** de `embedding_info.json`,
3. imprime un **informe de calibración** (Recall@3 de la búsqueda vectorial, BM25 e híbrida, por tipo de pregunta),
4. arma y descarga el `datos_examen.zip` final (11 archivos de datos + `LEEME.txt`).

**Cómo usarlo en Colab:** *Entorno de ejecución → Ejecutar todo*. Cuando la celda 2 lo pida, suba `datos_examen_PARCIAL_sin_embeddings.zip`. No necesita GPU.

## 1 · Instalación

In [ ]:
%pip install -q sentence-transformers rank_bm25 pandas numpy

## 2 · Cargar `datos_examen/`

In [ ]:
import json, zipfile
from pathlib import Path
D = Path("datos_examen")
if not D.exists():
    try:
        from google.colab import files
        print("Suba datos_examen_PARCIAL_sin_embeddings.zip (o cualquier zip que contenga la carpeta datos_examen/)")
        subido = files.upload()
        zipfile.ZipFile(next(iter(subido))).extractall(".")
    except ImportError:
        pass
assert D.exists(), "No se encontró la carpeta datos_examen/. Suba el zip o coloque la carpeta junto al notebook."

info = json.load(open(D / "embedding_info.json", encoding="utf-8"))
docs = json.load(open(D / "corpus_reglamento.json", encoding="utf-8"))
qs   = json.load(open(D / "preguntas_eval.json", encoding="utf-8"))
print("Modelo oficial:", info["model"])
print("Documentos:", len(docs), "| Preguntas:", len(qs))
print("Archivos presentes:", sorted(p.name for p in D.iterdir()))

## 3 · Generar los embeddings
Se embebe **solo** el campo `content` de cada documento y el campo `question` de cada pregunta, normalizados (norma 1). Esto es lo que dice `embedding_info.json` y lo que el estudiante debe reproducir en C.0.2.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

st  = SentenceTransformer(info["model"])
enc = lambda textos: np.asarray(st.encode(textos, normalize_embeddings=True), dtype="float32")
E = enc([d["content"] for d in docs])
Q = enc([q["question"] for q in qs])

assert E.shape[0] == len(docs) and Q.shape[0] == len(qs)
assert np.allclose(np.linalg.norm(E, axis=1), 1, atol=1e-3) and np.allclose(np.linalg.norm(Q, axis=1), 1, atol=1e-3)
np.save(D / "corpus_embeddings.npy", E)
np.save(D / "question_embeddings.npy", Q)
info["dimension"] = int(E.shape[1])
json.dump(info, open(D / "embedding_info.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("corpus_embeddings:", E.shape, "| question_embeddings:", Q.shape, "| dimensión:", info["dimension"])

# Reproducibilidad: recalcular un documento debe dar el mismo vector (es lo que verifica el estudiante en C.0.2)
v = enc([docs[0]["content"]])[0]
print("Coseno recálculo doc 0 vs guardado:", float(v @ E[0]))

## 4 · Calibración
Se espera: **BM25 fuerte en `sigla`**, **vectorial fuerte en `parafrasis`** y **la híbrida más estable**. En una medición previa (solo BM25) el resultado fue 6/6 en sigla y 2/6 en paráfrasis.
Si la vectorial **no** supera a BM25 en paráfrasis, cambie esas preguntas en `preguntas_eval.json` antes de aplicar el examen (C.2 pide interpretar ese patrón).

In [ ]:
import re, pandas as pd
from rank_bm25 import BM25Okapi

ids = [d["id"] for d in docs]
tok = lambda t: re.findall(r"\w+", t.lower())
bm  = BM25Okapi([tok(d["title"] + " " + d["content"]) for d in docs])

def rrf(listas, k=60):
    s = {}
    for lst in listas:
        for r, d in enumerate(lst): s[d] = s.get(d, 0) + 1 / (k + r + 1)
    return sorted(s, key=s.get, reverse=True)

filas = []
for i, q in enumerate(qs):
    if q["tipo"] not in ("sigla", "parafrasis"): continue
    r_vec = [ids[j] for j in np.argsort(-(E @ Q[i]))]
    r_bm  = [ids[j] for j in np.argsort(-bm.get_scores(tok(q["question"])))]
    r_hib = rrf([r_vec, r_bm])
    g = q["gold_doc_id"]
    filas.append({"tipo": q["tipo"], "pregunta": q["question"], "gold": g,
                  "vectorial@3": g in r_vec[:3], "bm25@3": g in r_bm[:3], "hibrida@3": g in r_hib[:3],
                  "rank_vectorial": r_vec.index(g) + 1})
df = pd.DataFrame(filas)
resumen = df.groupby("tipo")[["vectorial@3", "bm25@3", "hibrida@3"]].agg(["sum", "count"])
print("Recall@3 por tipo (aciertos / total):")
for tipo, sub in df.groupby("tipo"):
    print(f"  {tipo:11s}", {m: f"{int(sub[m].sum())}/{len(sub)}" for m in ["vectorial@3", "bm25@3", "hibrida@3"]})
print(f"MRR vectorial (12 preguntas): {np.mean(1 / df['rank_vectorial']):.3f}")
df[["tipo", "pregunta", "gold", "vectorial@3", "bm25@3", "hibrida@3", "rank_vectorial"]]

In [ ]:
# Preguntas donde falla la vectorial (para decidir si hay que reescribirlas)
fallos = df[~df["vectorial@3"]][["tipo", "pregunta", "gold", "rank_vectorial"]]
print("Sin fallos de la vectorial en top-3 ✅" if fallos.empty else f"La vectorial falla en {len(fallos)} pregunta(s):")
fallos

## 5 · Armar y descargar `datos_examen.zip`

In [ ]:
esperados = ["yolo_results_scratch.csv", "yolo_results_pretrained.csv", "yolo_results_freeze.csv",
             "yolo_test_preds.json", "yolo_test_gt.json", "embedding_info.json", "corpus_reglamento.json",
             "preguntas_eval.json", "corpus_embeddings.npy", "question_embeddings.npy", "faithfulness_cases.json"]
faltan = [f for f in esperados if not (D / f).exists()]
assert not faltan, f"Faltan archivos: {faltan}"

with zipfile.ZipFile("datos_examen.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(D.iterdir()):
        z.write(f, arcname=f"datos_examen/{f.name}")
print("✅ datos_examen.zip creado con", len(zipfile.ZipFile("datos_examen.zip").namelist()), "archivos")

try:
    from google.colab import files
    files.download("datos_examen.zip")
except ImportError:
    print("Fuera de Colab: el zip quedó junto al notebook.")

### Antes de publicar
- [ ] La vectorial gana (o empata) en `parafrasis` y BM25 en `sigla`; si no, reescriba las preguntas.
- [ ] Los archivos YOLO son **simulados** hasta que ejecute `generar_yolo_real.py`; avise al grupo si los deja así.
- [ ] Suba el zip a Drive y pruébelo en un Colab limpio.